In [2]:
import json
import random

import numpy as np
import pandas as pd
import networkx as nx

from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity

##Exercise 1. Normalizing Actor-Genre Matrix
This exercise builds on last week's but asks the student apply L1 normalization to the rows of the matrix. 

1) Download imdb_movies_2000to2022.prolific.jsonLinks to an external site. from GitHub (also available on Canvas in Exercises/data)
2) Create a data frame, where each row corresponds to an actor, each column represents a genre, and each cell captures how many times that row’s actor has appeared in that column’s genre
3) Using this data frame as your “feature matrix”, for every row in the matrix, apply L1 normalization. That is, calculate the sum of each row, and divide every element of that row by this sum. Store this normalized matrix as a new object. If you apply `sum(axis=1)` to the normalized matrix, all actors should now have a value of 1.
4) Using this L1-normalized data frame as your new feature matrix, select an actor (called your “query”) for whom you want to find the top 10 most similar actors based on the genres in which they’ve starred
As an example, select the row from your data frame associated with Chris Hemsworth, actor ID “nm1165110”, as your “query” actor
5) Use sklearn.metrics.DistanceMetricLinks to an external site. to calculate the Euclidean distances between your query actor and all other actors based on this normalized matrix of genre appearances.
6) Print a list of the top ten actors most similar to your query actor using Euclidean distance
7) Describe how this list has changed compared to Cosine Similarity.

In [3]:
actor_genres = {}
with open("imdb_movies_2000to2022.prolific.json") as f:
    for line in f:
        movie = json.loads(line)
        genres = movie["genres"]
        actors = movie["actors"]
        for actor_id, actor_name in actors:
            if not actor_id in actor_genres:
                actor_genres[actor_id] = {}
            for genre in genres:
                if not genre in actor_genres[actor_id]:
                    actor_genres[actor_id][genre] = 0
                actor_genres[actor_id][genre] += 1

actor_genres_df = pd.DataFrame.from_dict(actor_genres, orient="index")
actor_genres_df = actor_genres_df.fillna(0)
actor_genres_df.head()

,Comedy,Fantasy,Romance,Drama,Mystery,Thriller,Action,Biography,Crime,War,...,Horror,Documentary,Sport,News,Family,Music,,Western,Short,Reality-TV
nm0000212,7.0,1.0,6.0,6.0,1.0,2.0,1.0,1.0,2.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nm0413168,7.0,3.0,5.0,12.0,5.0,2.0,14.0,4.0,6.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nm0000630,8.0,2.0,6.0,14.0,2.0,3.0,4.0,5.0,1.0,1.0,...,3.0,7.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
nm0005227,10.0,1.0,2.0,2.0,0.0,1.0,1.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
nm0864851,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
actor_genres_df_normalized = actor_genres_df.div(actor_genres_df.sum(axis=1), axis=0)
actor_genres_df_normalized.head()

,Comedy,Fantasy,Romance,Drama,Mystery,Thriller,Action,Biography,Crime,War,...,Horror,Documentary,Sport,News,Family,Music,,Western,Short,Reality-TV
nm0000212,0.250000,0.035714,0.214286,0.214286,0.035714,0.071429,0.035714,0.035714,0.071429,0.035714,...,0.000000,0.000000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0
nm0413168,0.081395,0.034884,0.058140,0.139535,0.058140,0.023256,0.162791,0.046512,0.069767,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0
nm0000630,0.112676,0.028169,0.084507,0.197183,0.028169,0.042254,0.056338,0.070423,0.014085,0.014085,...,0.042254,0.098592,0.042254,0.014085,0.00,0.0,0.0,0.0,0.0,0.0
nm0005227,0.400000,0.040000,0.080000,0.080000,0.000000,0.040000,0.040000,0.000000,0.000000,0.000000,...,0.040000,0.000000,0.040000,0.000000,0.08,0.0,0.0,0.0,0.0,0.0
nm0864851,0.333333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.333333,0.000000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0


In [5]:
query_actor = "nm1165110"
top_actors_to_print = 10
query_actor_genres = actor_genres_df_normalized.loc[query_actor]

distances = euclidean_distances(query_actor_genres.values.reshape(1, -1), actor_genres_df_normalized.values)

#print top actors similar
top_actors = np.argsort(distances)[0][:top_actors_to_print]
for i, actor in enumerate(actor_genres_df_normalized.index[top_actors]):
    print(f"{i+1}. {actor} - {distances[0][top_actors[i]]}")

1. nm1165110 - 0.0
2. nm0000129 - 0.09789754549980856
3. nm0829032 - 0.12496192961003834
4. nm0147147 - 0.12797458635598938
5. nm5899377 - 0.134492704809032
6. nm0003244 - 0.13472631787668812
7. nm1679372 - 0.13596547538990353
8. nm2018237 - 0.14047239052070512
9. nm4043618 - 0.14238483556047654
10. nm2207222 - 0.14953462799296718


In [6]:
#cosine similarity
cosine_sim = cosine_similarity(query_actor_genres.values.reshape(1, -1), actor_genres_df_normalized.values)
top_actors = np.argsort(cosine_sim[0])[::-1][:top_actors_to_print]
for i, actor in enumerate(actor_genres_df_normalized.index[top_actors]):
    print(f"{i+1}. {actor} - {cosine_sim[0][top_actors[i]]}")

1. nm1165110 - 1.0
2. nm0000129 - 0.9731153693864449
3. nm0147147 - 0.9554009262802291
4. nm0829032 - 0.9547966369872284
5. nm5899377 - 0.9512174592103435
6. nm1679372 - 0.9483057620304723
7. nm0003244 - 0.9465853865185674
8. nm0636280 - 0.943474290675864
9. nm0607884 - 0.943474290675864
10. nm2018237 - 0.9419962824696096


Most of the results in top 10 looks similar with a few exception. Overall results do make sense. 